# 04 - Results and Evaluation
Evaluate model outputs and inspect arbitrage signals with explainability diagnostics.

In [ ]:
from config import ARBITRAGE_WINDOW, PAIR_CONFIGS, VOLATILITY_FILTER_QUANTILE, ZSCORE_ENTRY, ZSCORE_EXIT
from src.arbitrage_detector import ArbitrageDetector
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer
from src.models import TrackingErrorModel

loader = MarketDataLoader()
panel = loader.fetch_universe(PAIR_CONFIGS, period='2y', interval='1d')
features = FeatureEngineer(rolling_window=20, horizon=1).transform_universe(panel)
model = TrackingErrorModel.load('artifacts/te_model.joblib')
detector = ArbitrageDetector(
    window=ARBITRAGE_WINDOW,
    zscore_entry=ZSCORE_ENTRY,
    zscore_exit=ZSCORE_EXIT,
    volatility_filter_quantile=VOLATILITY_FILTER_QUANTILE,
)

features_clean = features.dropna()
x = features_clean[model.numeric_columns + model.categorical_columns]
pred = model.predict(x)
pred[:5]

In [ ]:
pair_name = features_clean['pair'].iloc[-1]
pair_panel = panel[panel['pair'] == pair_name]
signal_snapshot = detector.latest_signal(pair_panel)
signal_snapshot

In [ ]:
shap_df = model.explain_shap(x.tail(200), max_samples=100).head(10)
shap_df